# Baseline Model Evaluation & Analysis

This notebook evaluates the best trained model on the test set and generates analysis plots (confusion matrix, t-SNE).

**Targets (Reference Paper):**
- Sensitivity (Se): 68.31%
- Specificity (Sp): 67.89%
- ICBHI Score: 68.10%

**Acceptable thresholds for baseline:**
- Se ≥ 64%
- Sp ≥ 60%
- Score ≥ 62%

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# Project imports
from src.dataset import ICBHIDataset
from src.model import load_model
from src.evaluate import compute_icbhi_metrics
import yaml

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ Imports successful")
print(f"🖥️ Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

In [ ]:
# Load config
with open("../configs/baseline.yaml") as f:
    config = yaml.safe_load(f)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load best model
checkpoint_path = "../checkpoints/best_model.pt"
model = load_model(config, pretrained=True, freeze=False)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()
print(f"✅ Model loaded from {checkpoint_path}")
print(f"   Epochs trained: {checkpoint.get('epoch', 'N/A')}")
print(f"   Best Se: {checkpoint.get('best_Se', 'N/A'):.4f}")

# Load test dataset
test_dataset = ICBHIDataset("../data/splits/test.csv", config, augment=False)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)
print(f"✅ Test dataset loaded: {len(test_dataset)} cycles")

In [ ]:
# Run inference on test set
all_preds = []
all_labels = []
all_embeddings = []

print("Running inference on test set...")
with torch.no_grad():
    for specs, labels in tqdm(test_loader, desc="Evaluating"):
        specs = specs.to(device)
        labels = labels.cpu().numpy()
        
        # Forward pass
        outputs = model(specs)  # logits: (B, 4)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(f"✅ Inference complete")
print(f"   Total samples: {len(all_labels)}")
print(f"   Predictions shape: {all_preds.shape}")
print(f"   Labels shape: {all_labels.shape}")

In [ ]:
# Compute ICBHI metrics (binary: normal vs abnormal)
metrics = compute_icbhi_metrics(all_labels, all_preds)

print("\n" + "="*60)
print("BASELINE RESULTS (AST+SAM)")
print("="*60)
print(f"Sensitivity (Se):    {metrics['Se']:.4f}  ({metrics['Se']*100:.2f}%)")
print(f"Specificity (Sp):    {metrics['Sp']:.4f}  ({metrics['Sp']*100:.2f}%)")
print(f"ICBHI Score:         {metrics['Score']:.4f}  ({metrics['Score']*100:.2f}%)")
print("="*60)

# Compare with reference paper
ref_se = 0.6831
ref_sp = 0.6789
ref_score = 0.6810

print("\nCOMPARISON WITH REFERENCE PAPER:")
print(f"  Se:    {metrics['Se']*100:.2f}% vs {ref_se*100:.2f}% (Δ {(metrics['Se']-ref_se)*100:+.2f}%)")
print(f"  Sp:    {metrics['Sp']*100:.2f}% vs {ref_sp*100:.2f}% (Δ {(metrics['Sp']-ref_sp)*100:+.2f}%)")
print(f"  Score: {metrics['Score']*100:.2f}% vs {ref_score*100:.2f}% (Δ {(metrics['Score']-ref_score)*100:+.2f}%)")

# Save results
results_path = Path("../logs/baseline_results.txt")
results_path.parent.mkdir(parents=True, exist_ok=True)

with open(results_path, "w") as f:
    f.write("Baseline AST+SAM\n")
    f.write("="*60 + "\n")
    f.write(f"Sensitivity (Se):    {metrics['Se']*100:.2f}%\n")
    f.write(f"Specificity (Sp):    {metrics['Sp']*100:.2f}%\n")
    f.write(f"ICBHI Score:         {metrics['Score']*100:.2f}%\n")
    f.write("="*60 + "\n")
    f.write(f"Epochs trained:      20\n")
    f.write(f"Batch size:          16\n")
    f.write(f"Test samples:        {len(all_labels)}\n")
    f.write(f"\nReference Paper Targets:\n")
    f.write(f"  Se:    {ref_se*100:.2f}%\n")
    f.write(f"  Sp:    {ref_sp*100:.2f}%\n")
    f.write(f"  Score: {ref_score*100:.2f}%\n")

print(f"\n✅ Results saved to {results_path}")

In [ ]:
# Generate 4-class confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

CLASS_NAMES = ["Normal", "Crackle", "Wheeze", "Both"]

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1, 2, 3])

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title("Baseline AST+SAM — 4-Class Confusion Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()

cm_path = Path("../logs/confusion_matrix_baseline.png")
plt.savefig(cm_path, dpi=150, bbox_inches="tight")
print(f"✅ Confusion matrix saved to {cm_path}")
plt.show()

# Print per-class metrics
print("\nPer-Class Metrics (4-class):")
print(f"{'Class':<10} {'Support':<10} {'Precision':<12} {'Recall':<12}")
print("-"*50)
for i, class_name in enumerate(CLASS_NAMES):
    support = (all_labels == i).sum()
    if support == 0:
        continue
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"{class_name:<10} {support:<10} {precision:<12.4f} {recall:<12.4f}")

In [ ]:
# Extract embeddings from penultimate layer for t-SNE
print("Extracting embeddings for t-SNE (this may take a minute)...")

# Create a hook to extract embeddings
embeddings_list = []

def hook_fn(module, input, output):
    # output is the penultimate representation before classification head
    embeddings_list.append(output.detach().cpu().numpy())

# Register hook on the last hidden layer before classification
# For AST model from transformers, we hook into the classifier or pooler
if hasattr(model, 'classifier'):
    model.classifier.register_forward_hook(hook_fn)
elif hasattr(model, 'pooler'):
    model.pooler.register_forward_hook(hook_fn)
else:
    # Fallback: try to hook the main body
    print("Warning: Could not find classifier or pooler. Using forward pass output.")

# Extract embeddings
with torch.no_grad():
    for specs, labels in tqdm(test_loader, desc="Extracting embeddings"):
        specs = specs.to(device)
        _ = model(specs)

if embeddings_list:
    embeddings = np.concatenate(embeddings_list, axis=0)
    print(f"✅ Embeddings extracted: shape {embeddings.shape}")
else:
    print("⚠️ Could not extract embeddings via hook. Using model output logits instead.")
    embeddings = None

In [ ]:
# Generate t-SNE plot
from sklearn.manifold import TSNE

if embeddings is not None:
    print("Computing t-SNE (this may take 1-2 minutes)...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
    embeddings_tsne = tsne.fit_transform(embeddings)
    print("✅ t-SNE computed")
else:
    print("Skipping t-SNE due to missing embeddings")
    embeddings_tsne = None

# Plot t-SNE
if embeddings_tsne is not None:
    fig, ax = plt.subplots(figsize=(12, 10))
    
    colors = {0: "#1f77b4", 1: "#ff7f0e", 2: "#2ca02c", 3: "#d62728"}
    labels_map = {0: "Normal", 1: "Crackle", 2: "Wheeze", 3: "Both"}
    
    for class_id in range(4):
        mask = all_labels == class_id
        ax.scatter(
            embeddings_tsne[mask, 0],
            embeddings_tsne[mask, 1],
            c=colors[class_id],
            label=labels_map[class_id],
            alpha=0.6,
            s=30,
            edgecolors="none"
        )
    
    ax.set_xlabel("t-SNE 1", fontsize=12)
    ax.set_ylabel("t-SNE 2", fontsize=12)
    ax.set_title("Baseline AST+SAM — t-SNE Embedding Visualization", fontsize=14, fontweight="bold")
    ax.legend(fontsize=11, loc="best")
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    tsne_path = Path("../logs/tsne_baseline.png")
    plt.savefig(tsne_path, dpi=150, bbox_inches="tight")
    print(f"✅ t-SNE plot saved to {tsne_path}")
    plt.show()

In [ ]:
# Summary & Next Steps Analysis
print("\n" + "="*70)
print("PHASE 4 COMPLETION & PHASE 5 DECISION")
print("="*70)

# Thresholds
THRESHOLD_SE = 0.64
THRESHOLD_SP = 0.60
THRESHOLD_SCORE = 0.62

se = metrics['Se']
sp = metrics['Sp']
score = metrics['Score']

# Determine pass/fail status
se_pass = se >= THRESHOLD_SE
sp_pass = sp >= THRESHOLD_SP
score_pass = score >= THRESHOLD_SCORE

print("\n✅ BASELINE VERIFICATION:")
print(f"  [{'✓' if se_pass else '✗'}] Sensitivity:  {se*100:.2f}% {'≥' if se_pass else '<'} {THRESHOLD_SE*100:.0f}%")
print(f"  [{'✓' if sp_pass else '✗'}] Specificity:  {sp*100:.2f}% {'≥' if sp_pass else '<'} {THRESHOLD_SP*100:.0f}%")
print(f"  [{'✓' if score_pass else '✗'}] Score:       {score*100:.2f}% {'≥' if score_pass else '<'} {THRESHOLD_SCORE*100:.0f}%")

print("\n📊 FAILURE MODE ANALYSIS (from confusion matrix):")
# Analyze what the model is confusing
print("\nMisclassifications:")
for i in range(4):
    for j in range(4):
        if i != j and cm[i, j] > 0:
            pct = (cm[i, j] / cm[i, :].sum()) * 100
            print(f"  {CLASS_NAMES[i]:<10} → {CLASS_NAMES[j]:<10}: {cm[i, j]:>3} ({pct:>5.1f}%)")

print("\n🎯 NEXT STEPS:")
if se_pass and sp_pass and score_pass:
    print("✅ BASELINE REPRODUCED — Ready for PHASE 5 (Improvement Experiments)")
    print("   Recommendations:")
    print("   1. Try Focal Loss (γ=2) for minority class boosting")
    print("   2. Experiment with stronger augmentation (SpecAugment)")
    print("   3. Consider threshold tuning for Se optimization")
    print("   4. Try ensemble methods if individual Se < 70%")
elif se < THRESHOLD_SE:
    print(f"⚠️  LOW SENSITIVITY ({se*100:.2f}% < {THRESHOLD_SE*100:.0f}%)")
    print("   Debugging steps:")
    print("   1. Verify patient-level split has zero overlap")
    print("   2. Check input shape transformation: (B,1,128,T) → (B,T,128)")
    print("   3. Verify cyclic padding (not zero-padding) is used")
    print("   4. Check weighted sampler is active during training")
    print("   5. Review confusion matrix: are abnormal samples being misclassified as normal?")
else:
    print(f"✓  Metrics acceptable (Se={se*100:.2f}%, Sp={sp*100:.2f}%, Score={score*100:.2f}%)")
    print("   Ready for PHASE 5 improvement experiments")

print("\n" + "="*70)